# Confusion Matrix

This notebook builds the confusion matrix for the current saved checkpoint and exports a PNG for presentation use.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix, roc_curve, roc_auc_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.inference.resume_screener import ResumeScreeningModel
from ml.training.train_model import DEFAULT_THRESHOLD, load_processed_tensors

processed_dir = PROJECT_ROOT / "data" / "processed"
model_path = PROJECT_ROOT / "ml" / "models" / "resume_net.pt"
plots_dir = PROJECT_ROOT / "notebooks" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

_, X_test, _, y_test = load_processed_tensors(processed_dir=processed_dir)
model = ResumeScreeningModel(str(model_path), device="cpu", threshold=DEFAULT_THRESHOLD)

with torch.no_grad():
    probabilities = torch.sigmoid(model.model(X_test)).detach().cpu().numpy().reshape(-1)

labels = y_test.cpu().numpy().reshape(-1).astype(int)
predictions = (probabilities >= DEFAULT_THRESHOLD).astype(int)

print(f"Threshold: {DEFAULT_THRESHOLD}")
print(f"Test size: {labels.shape[0]}")


In [ ]:
cm = confusion_matrix(labels, predictions)
fig, ax = plt.subplots(figsize=(6, 5))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Reject (0)", "Shortlist (1)"])
display.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix at Threshold {DEFAULT_THRESHOLD}")
fig.tight_layout()
output_path = plots_dir / "confusion_matrix.png"
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()
tn, fp, fn, tp = cm.ravel()
print({"true_negative": int(tn), "false_positive": int(fp), "false_negative": int(fn), "true_positive": int(tp)})
print(f"Saved plot to: {output_path}")
